# LAB 10 — Fine-tuning & Controlled Experiments (Experiments 1–5)
Notebook huấn luyện mô hình `ExtractiveQAModel` trên SQuAD v1.1, đánh giá EM/F1, chạy ablation study về `max_length`, phân tích độ dài đáp án và xuất toàn bộ biểu đồ.

In [1]:
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from qa_model import (
    ExtractiveQAModel,
    keyword_overlap_baseline,
    compute_exact_match,
    compute_f1,
    decode_best_span,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Using device:", DEVICE)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Using device: cuda


In [2]:
# Tải SQuAD v1.1
raw_datasets = load_dataset("rajpurkar/squad")
train_raw = raw_datasets["train"].shuffle(seed=SEED).select(range(15000))
val_raw = raw_datasets["validation"].shuffle(seed=SEED).select(range(2000))

# Đánh giá Heuristic Baseline (Mục 21)
baseline_ems, baseline_f1s = [], []
for ex in val_raw:
    pred = keyword_overlap_baseline(ex["question"], ex["context"])
    golds = ex["answers"]["text"]
    baseline_ems.append(max(compute_exact_match(pred, g) for g in golds))
    baseline_f1s.append(max(compute_f1(pred, g) for g in golds))

print(f"Heuristic Baseline -> EM: {np.mean(baseline_ems)*100:.2f}% | F1: {np.mean(baseline_f1s)*100:.2f}%")

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Heuristic Baseline -> EM: 1.00% | F1: 18.95%


In [3]:
def prepare_train_features(examples, max_length=384, stride=128):
    questions = [q.strip() for q in examples["question"]]
    tokenized_examples = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")
    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized_examples.sequence_ids(i)
        answers = examples["answers"][sample_mapping[i]]

        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
            continue

        answer_start = answers["answer_start"][0]
        answer_end = answer_start + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1
        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if not (offsets[token_start_index][0] <= answer_start and offsets[token_end_index][1] >= answer_end):
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= answer_start:
                token_start_index += 1
            start_token = token_start_index - 1
            while offsets[token_end_index][1] >= answer_end:
                token_end_index -= 1
            end_token = token_end_index + 1
            tokenized_examples["start_positions"].append(start_token)
            tokenized_examples["end_positions"].append(end_token)

    return tokenized_examples


def evaluate_model(model, dataset_split, max_length=384, stride=128):
    model.eval()
    records = []
    with torch.no_grad():
        for ex in dataset_split:
            enc = tokenizer(
                ex["question"].strip(),
                ex["context"],
                truncation="only_second",
                max_length=max_length,
                stride=stride,
                return_overflowing_tokens=True,
                return_offsets_mapping=True,
                padding="max_length",
                return_tensors="pt",
            )
            offsets_all = enc.pop("offset_mapping")
            enc.pop("overflow_to_sample_mapping")

            best_pred, best_score = "", -1e9
            out = model(enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE))
            for w in range(enc["input_ids"].size(0)):
                pred_w, score_w = decode_best_span(
                    out["start_logits"][w],
                    out["end_logits"][w],
                    offsets_all[w].tolist(),
                    enc.sequence_ids(w),
                    ex["context"],
                )
                if score_w > best_score:
                    best_score = score_w
                    best_pred = pred_w

            golds = ex["answers"]["text"]
            em = max(compute_exact_match(best_pred, g) for g in golds)
            f1 = max(compute_f1(best_pred, g) for g in golds)
            gold_len = len(tokenizer.tokenize(golds[0]))
            records.append({
                "question": ex["question"],
                "gold": golds[0],
                "prediction": best_pred,
                "em": em,
                "f1": f1,
                "gold_len": gold_len,
            })
    return pd.DataFrame(records)

In [4]:
def train_qa(max_length=384, stride=128, epochs=3, batch_size=16, lr=3e-5):
    train_ds = train_raw.map(
        lambda x: prepare_train_features(x, max_length=max_length, stride=stride),
        batched=True,
        remove_columns=train_raw.column_names,
    )
    train_ds.set_format("torch")
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = ExtractiveQAModel(MODEL_NAME).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                out = model(
                    input_ids=batch["input_ids"].to(DEVICE),
                    attention_mask=batch["attention_mask"].to(DEVICE),
                    start_positions=batch["start_positions"].to(DEVICE),
                    end_positions=batch["end_positions"].to(DEVICE),
                )
                loss = out["loss"]
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item()

        eval_df = evaluate_model(model, val_raw, max_length=max_length, stride=stride)
        print(
            f"[max_len={max_length}] Epoch {epoch}/{epochs} | "
            f"Train Loss: {running_loss/len(train_loader):.4f} | "
            f"Val EM: {eval_df['em'].mean()*100:.2f}% | Val F1: {eval_df['f1'].mean()*100:.2f}%"
        )

    elapsed = time.time() - t0
    return model, eval_df, elapsed


In [5]:
# Reproducible execution for Experiments 1-5. Logs and artifacts are written to artifacts/.
from pathlib import Path
import json, os
OUT = Path("artifacts")
OUT.mkdir(exist_ok=True)
all_results = []
all_eval = {}
for length, stride in [(128, 64), (256, 96), (384, 128)]:
    print(f"\n=== Experiment 1/2: max_length={length} ===", flush=True)
    model, eval_df, elapsed = train_qa(max_length=length, stride=stride, epochs=3, batch_size=16, lr=3e-5)
    eval_df.to_csv(OUT / f"predictions_maxlen_{length}.csv", index=False)
    torch.save(model.state_dict(), OUT / f"qa_maxlen_{length}.pt")
    row = {"experiment":"context_length", "max_length":length, "stride":stride, "epochs":3,
           "train_samples":len(train_raw), "validation_samples":len(val_raw), "train_seconds":elapsed,
           "em":float(eval_df.em.mean()), "f1":float(eval_df.f1.mean())}
    all_results.append(row); all_eval[length]=eval_df
    pd.DataFrame(all_results).to_csv(OUT / "results.csv", index=False)

# Experiments 3-5 use the best context-length model (selected by validation F1).
best = max(all_results, key=lambda x:x["f1"])
best_eval = all_eval[best["max_length"]]
baseline = pd.DataFrame([{"question":e["question"], "gold":e["answers"]["text"][0],
    "prediction":keyword_overlap_baseline(e["question"],e["context"]),
    "f1":max(compute_f1(keyword_overlap_baseline(e["question"],e["context"]),g) for g in e["answers"]["text"]),
    "em":max(compute_exact_match(keyword_overlap_baseline(e["question"],e["context"]),g) for g in e["answers"]["text"])} for e in val_raw])
baseline.to_csv(OUT / "baseline_predictions.csv", index=False)
best_eval.to_csv(OUT / "best_predictions.csv", index=False)

# Experiment 4: partition by lexical overlap between question and gold answer.
import re
stop_words = {"what","where","when","who","whom","whose","which","why","how","is","are","was","were","the","a","an","of","in","to","for","and","on","at","by"}
def content_words(text): return set(re.findall(r"[a-z0-9]+", text.lower())) - stop_words
lex = []
for _, row in best_eval.iterrows():
    q, a = content_words(row.question), content_words(row.gold)
    ratio = len(q & a) / max(1, len(a))
    lex.append("high_overlap" if ratio >= .5 else "low_overlap")
best_eval.assign(lexical_group=lex).groupby("lexical_group").agg(n=("em","size"),em=("em","mean"),f1=("f1","mean")).to_csv(OUT/"experiment4_lexical_overlap.csv")

# Experiment 5: answer length buckets.
def length_bucket(n):
    return "1" if n==1 else "2-3" if n<=3 else "4-6" if n<=6 else "7+"
best_eval.assign(answer_length_group=best_eval.gold_len.map(length_bucket)).groupby("answer_length_group").agg(n=("em","size"),em=("em","mean"),f1=("f1","mean")).to_csv(OUT/"experiment5_answer_length.csv")

# Experiment 3: aggregate model vs baseline and collect partial-match examples.
summary = [{"model":"keyword_overlap_baseline","em":float(baseline.em.mean()),"f1":float(baseline.f1.mean())},
           {"model":f"distilbert_maxlen_{best['max_length']}","em":float(best_eval.em.mean()),"f1":float(best_eval.f1.mean())}]
pd.DataFrame(summary).to_csv(OUT/"experiment3_em_f1.csv", index=False)
best_eval[(best_eval.em==0)&(best_eval.f1>0)].head(15).to_csv(OUT/"partial_match_examples.csv",index=False)
print("\nEXPERIMENT_SUMMARY", json.dumps({"runs":all_results,"best":best,"summary":summary},indent=2), flush=True)



=== Experiment 1/2: max_length=128 ===


Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


/tmp/ipykernel_2181673/591567009.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_2181673/591567009.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


... (14 outputs left out)

[max_len=384] Epoch 1/3 | Train Loss: 2.4585 | Val EM: 64.05% | Val F1: 74.69%


[max_len=384] Epoch 2/3 | Train Loss: 1.1510 | Val EM: 67.15% | Val F1: 77.04%


[max_len=384] Epoch 3/3 | Train Loss: 0.7755 | Val EM: 67.05% | Val F1: 77.10%



EXPERIMENT_SUMMARY {
  "runs": [
    {
      "experiment": "context_length",
      "max_length": 128,
      "stride": 64,
      "epochs": 3,
      "train_samples": 15000,
      "validation_samples": 2000,
      "train_seconds": 552.4649000167847,
      "em": 0.6715,
      "f1": 0.770715956876333
    },
    {
      "experiment": "context_length",
      "max_length": 256,
      "stride": 96,
      "epochs": 3,
      "train_samples": 15000,
      "validation_samples": 2000,
      "train_seconds": 414.32516288757324,
      "em": 0.6695,
      "f1": 0.7672351251157696
    },
    {
      "experiment": "context_length",
      "max_length": 384,
      "stride": 128,
      "epochs": 3,
      "train_samples": 15000,
      "validation_samples": 2000,
      "train_seconds": 538.9815127849579,
      "em": 0.6705,
      "f1": 0.7710111847079697
    }
  ],
  "best": {
    "experiment": "context_length",
    "max_length": 384,
    "stride": 128,
    "epochs": 3,
    "train_samples": 15000,
    "valid